In [1]:
import torch
from dlordinal.datasets import FGNet
from torch import cuda
from torch.backends import mps
from torchvision.transforms.v2 import Compose, Normalize
from torchvision.transforms.v2 import Resize, RandomHorizontalFlip, CenterCrop

from util.image_experiment_runner import run_image_experiment

In [2]:


from torchvision.transforms.v2 import RandomResizedCrop, ColorJitter, RandomRotation, ToTensor

fgnet_train = FGNet(
    root="./datasets",
    download=True,
    train=True,
    transform=Compose([
        Resize(256),
        RandomResizedCrop(224, scale=(0.85, 1.0)),
        RandomHorizontalFlip(p=0.5),
        ColorJitter(
            brightness=0.2,
            contrast=0.2,
            saturation=0.2,
            hue=0.05
        ),
        RandomRotation(10),
        ToTensor(),
        Normalize(
            [0.485, 0.456, 0.406],
            [0.229, 0.224, 0.225]
        )
    ])
)

fgnet_test = FGNet(
    root="./datasets",
    download=True,
    train=False,
    transform=Compose([
        Resize(256),
        CenterCrop(224),
        ToTensor(),
        Normalize(
            [0.485, 0.456, 0.406],
            [0.229, 0.224, 0.225]
        )
    ])
)

num_classes = len(fgnet_train.classes)
device = torch.device("cuda" if cuda.is_available() else "mps" if mps.is_available() else "cpu")
print(f"Classes: {num_classes}, Device: {device}")

print(f"Train samples: {len(fgnet_train)}, Test samples: {len(fgnet_test)}")

Files already downloaded and verified
Files already processed and verified
Files already split and verified
Files already downloaded and verified
Files already processed and verified
Files already split and verified
Classes: 6, Device: cpu
Train samples: 801, Test samples: 201


In [ ]:
data_type = 'fgnet'
run_image_experiment(device, num_classes, data_type, fgnet_train, fgnet_test,
                     optimizer=torch.optim.AdamW, lr=2e-4, weight_decay=1e-4,
                     batch_size=32, es_patience=40, lr_patience=10, lr_factor=0.5,
                     lr_monitor="train_loss", stratified=False,
                     use_determinism=False, shuffle=False, aps_randomized=True)